In [1]:
from pyspark.sql.functions import col, to_timestamp

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
        .appName("Day3 Telematics Project") \
        .getOrCreate()

In [3]:
spark.conf.set("spark.sql.adaptive.enabled","false")

In [4]:
df_tele = spark.read.csv("/opt/spark-data/telematics.csv",header=True,inferSchema=True)
df_vehicle = spark.read.csv("/opt/spark-data/vehicles.csv",header=True,inferSchema=True)

In [5]:
df_tele.printSchema()
df_vehicle.show()
df_tele.show(5)

root
 |-- timestamp: timestamp (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- speed: integer (nullable = true)
 |-- engine_temp: integer (nullable = true)
 |-- fuel_level: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)

+----------+-----+-----+----+
|vehicle_id|model|plant|year|
+----------+-----+-----+----+
|        V1|Sedan|    A|2022|
|        V2|  SUV|    B|2023|
|        V3|Truck|    A|2021|
|        V4|Sedan|    C|2020|
|        V5|  SUV|    B|2022|
+----------+-----+-----+----+

+-------------------+----------+-----+-----------+----------+--------+--------+
|          timestamp|vehicle_id|speed|engine_temp|fuel_level|     lat|     lon|
+-------------------+----------+-----+-----------+----------+--------+--------+
|2024-03-01 00:00:00|        V1|   41|         84|        50|13.72701|77.92426|
|2024-03-01 00:00:00|        V2|   17|        105|        95|13.86915|77.90112|
|2024-03-01 00:00:00|        V3|   79|    

In [6]:
from pyspark.sql.functions import lit

df_big = df_tele
df_big = df_big.withColumn("dup_id",lit(0))
for i in range(20):   # increase multiplier gradually
    df_big = df_big.union(df_tele.withColumn("dup_id", lit(i)))

df_big.count()

907200

In [7]:
df_tele = df_big

In [8]:
df_tele = df_tele.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"))
)

In [9]:
from pyspark.sql.functions import when

df_tele = df_tele.withColumn(
    "speed_category",
    when(col("speed") < 40, "low")
    .when(col("speed") < 80, "medium")
    .otherwise("high")
)

df_tele = df_tele.withColumn(
    "engine_stress",
    col("engine_temp") * col("speed")
)

In [10]:
df_agg = df_tele.groupBy("vehicle_id").agg(
    {"speed": "avg", "fuel_level": "avg"}
)

df_agg.show()

+----------+------------------+------------------+
|vehicle_id|   avg(fuel_level)|        avg(speed)|
+----------+------------------+------------------+
|        V3| 55.39780092592593|  60.1712962962963|
|        V4| 55.29907407407408| 59.89189814814815|
|        V5| 54.89247685185185| 59.64039351851852|
|        V2|55.083333333333336|60.079166666666666|
|        V1|54.793055555555554| 59.90127314814815|
+----------+------------------+------------------+



In [11]:
df_joined = df_tele.join(df_vehicle, on="vehicle_id", how="inner")
df_joined.explain()

== Physical Plan ==
*(23) Project [vehicle_id#18, timestamp#637, speed#19, engine_temp#20, fuel_level#21, lat#22, lon#23, dup_id#115, speed_category#646, engine_stress#656, model#49, plant#50, year#51]
+- *(23) BroadcastHashJoin [vehicle_id#18], [vehicle_id#48], Inner, BuildRight, false
   :- Union
   :  :- *(1) Project [timestamp#17 AS timestamp#637, vehicle_id#18, speed#19, engine_temp#20, fuel_level#21, lat#22, lon#23, 0 AS dup_id#115, CASE WHEN (speed#19 < 40) THEN low WHEN (speed#19 < 80) THEN medium ELSE high END AS speed_category#646, (engine_temp#20 * speed#19) AS engine_stress#656]
   :  :  +- *(1) Filter isnotnull(vehicle_id#18)
   :  :     +- FileScan csv [timestamp#17,vehicle_id#18,speed#19,engine_temp#20,fuel_level#21,lat#22,lon#23] Batched: false, DataFilters: [isnotnull(vehicle_id#18)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/opt/spark-data/telematics.csv], PartitionFilters: [], PushedFilters: [IsNotNull(vehicle_id)], ReadSchema: struct<timestamp:timestam

In [12]:
from pyspark.sql.functions import broadcast

df_joined_broadcast = df_tele.join(
    broadcast(df_vehicle),
    on="vehicle_id",
    how="inner"
)

df_joined_broadcast.explain()

== Physical Plan ==
*(23) Project [vehicle_id#18, timestamp#637, speed#19, engine_temp#20, fuel_level#21, lat#22, lon#23, dup_id#115, speed_category#646, engine_stress#656, model#49, plant#50, year#51]
+- *(23) BroadcastHashJoin [vehicle_id#18], [vehicle_id#48], Inner, BuildRight, false
   :- Union
   :  :- *(1) Project [timestamp#17 AS timestamp#637, vehicle_id#18, speed#19, engine_temp#20, fuel_level#21, lat#22, lon#23, 0 AS dup_id#115, CASE WHEN (speed#19 < 40) THEN low WHEN (speed#19 < 80) THEN medium ELSE high END AS speed_category#646, (engine_temp#20 * speed#19) AS engine_stress#656]
   :  :  +- *(1) Filter isnotnull(vehicle_id#18)
   :  :     +- FileScan csv [timestamp#17,vehicle_id#18,speed#19,engine_temp#20,fuel_level#21,lat#22,lon#23] Batched: false, DataFilters: [isnotnull(vehicle_id#18)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/opt/spark-data/telematics.csv], PartitionFilters: [], PushedFilters: [IsNotNull(vehicle_id)], ReadSchema: struct<timestamp:timestam

In [15]:
import time

start = time.time()
df_tele.join(df_vehicle, "vehicle_id").count()
print("Shuffle Join Time:", time.time() - start)

start = time.time()
df_tele.join(broadcast(df_vehicle), "vehicle_id").count()
print("Broadcast Join Time:", time.time() - start)

Shuffle Join Time: 0.5715718269348145
Broadcast Join Time: 0.587454080581665


### Imputing disturbances/skewness in the dataset

In [16]:
df_big.groupBy("vehicle_id").count().show()

+----------+------+
|vehicle_id| count|
+----------+------+
|        V3|181440|
|        V4|181440|
|        V5|181440|
|        V2|181440|
|        V1|181440|
+----------+------+



In [17]:
from pyspark.sql.functions import when, rand

df_skewed = df_big.withColumn(
    "vehicle_id",
    when(rand() < 0.7, "V1").otherwise(df_big.vehicle_id)
)

In [28]:
df_skewed_part = df_skewed.repartition(5, "vehicle_id")

In [29]:
df_skewed_part.groupBy("vehicle_id").count().show()

+----------+------+
|vehicle_id| count|
+----------+------+
|        V5| 54363|
|        V4| 54240|
|        V3| 54327|
|        V2| 54361|
|        V1|689909|
+----------+------+



In [45]:
df_skewed_part.rdd.glom().map(len).collect()

[0, 54363, 108567, 54361, 689909]

In [35]:
spark.conf.get("spark.sql.adaptive.enabled")

'false'

In [36]:
import time

start = time.time()
df_skewed_part.join(df_vehicle, "vehicle_id").count()
print("Shuffle Join Time:", time.time() - start)

start = time.time()
df_skewed_part.join(broadcast(df_vehicle), "vehicle_id").count()
print("Broadcast Join Time:", time.time() - start)

Shuffle Join Time: 0.7851302623748779
Broadcast Join Time: 0.5465037822723389
